# Notebook 3: Model Training
Train all models: baselines, XGBoost, LSTM with attention, and weighted ensemble.

## 1. Setup & Load Features

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, mlflow
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
from src.features.feature_pipeline import load_features
data = load_features('brent')
train_df, val_df, test_df = data['train_df'], data['val_df'], data['test_df']
feature_cols = data['feature_cols']
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Features: {len(feature_cols)}")

## 2. Baseline Models

In [ ]:
from src.models.baseline import NaiveForecaster, ARIMAForecaster

naive = NaiveForecaster().fit(train_df['close'])
naive_metrics = naive.evaluate_walk_forward(
    pd.concat([train_df['close'].iloc[-252:], val_df['close']]),
    horizon=1, min_train=200
)
print("Naïve baseline:")
for k, v in naive_metrics.items():
    if isinstance(v, float): print(f"  {k:<20} {v:.4f}")

arima = ARIMAForecaster(order=(2,1,2))
arima_metrics = arima.evaluate_walk_forward(
    pd.concat([train_df['close'].iloc[-504:], val_df['close']]),
    horizon=1, min_train=252, step=10
)
print("\nARIMA(2,1,2):")
for k, v in arima_metrics.items():
    if isinstance(v, float): print(f"  {k:<20} {v:.4f}")

## 3. XGBoost — Walk-Forward Validation

In [ ]:
mlflow.set_experiment('Oil_Price_Forecasting_Notebook')
from src.models.xgboost_model import XGBoostOilForecaster

with mlflow.start_run(run_name='xgboost_notebook'):
    model = XGBoostOilForecaster(reg_params={'n_estimators': 200})
    model.fit(train_df, val_df, feature_cols)
    metrics = model.evaluate(test_df, test_df['close'])
    mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, float)})

print("XGBoost test metrics:")
for k, v in metrics.items():
    if isinstance(v, float): print(f"  {k:<30} {v:.4f}")

## 4. XGBoost — Feature Importance & SHAP

In [ ]:
import shap

shap_df = model.explain(test_df, max_samples=300)
top15 = shap_df.abs().mean().nlargest(15)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# XGBoost feature importance
importances = pd.Series(model.regressor.feature_importances_, index=feature_cols).nlargest(15)
importances.sort_values().plot(kind='barh', ax=axes[0], color='#378ADD')
axes[0].set_title('XGBoost Feature Importance (top 15)')
axes[0].set_xlabel('Importance score')

# SHAP mean absolute value
top15.sort_values().plot(kind='barh', ax=axes[1], color='#EF9F27')
axes[1].set_title('SHAP Mean |Value| (top 15)')
axes[1].set_xlabel('Mean |SHAP value|')

plt.tight_layout()
plt.savefig('../docs/images/xgb_feature_importance.png', bbox_inches='tight')
plt.show()

print("\nTop 5 drivers for current forecast:")
for d in model.get_top_drivers(test_df.iloc[-1], n=5):
    print(f"  {d['factor'][:35]:<35} {d['impact']:<10} SHAP={d['shap_value']:+.5f}")

## 5. LSTM with Temporal Attention

In [ ]:
from src.models.lstm_model import LSTMTrainer, OilPriceDataset
from torch.utils.data import DataLoader

lstm_cols = train_df[feature_cols].var().nlargest(30).index.tolist()
print(f"LSTM features: {len(lstm_cols)}")

train_ds = OilPriceDataset(train_df, lstm_cols, lookback=60)
val_ds   = OilPriceDataset(val_df,   lstm_cols, lookback=60)
test_ds  = OilPriceDataset(test_df,  lstm_cols, lookback=60)

trainer = LSTMTrainer(n_features=len(lstm_cols), lookback=60)
trainer.feature_cols = lstm_cols

with mlflow.start_run(run_name='lstm_notebook'):
    trainer.fit(train_ds, val_ds, epochs=40, lr=5e-4, batch_size=64, patience=10, verbose=True)
    lstm_metrics = trainer.evaluate(test_ds)
    mlflow.log_metrics({k: v for k, v in lstm_metrics.items() if isinstance(v, float)})

print("\nLSTM metrics:", lstm_metrics)

## 6. LSTM Training Curve & Attention Weights

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(trainer.history['train_loss'], label='Train', color='#378ADD')
if trainer.history.get('val_loss'):
    axes[0].plot(trainer.history['val_loss'], label='Val', color='#D85A30')
axes[0].set_title('LSTM Training Loss (Huber)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Attention weights for last test sequence
last_seq = test_df[lstm_cols].iloc[-60:].replace([float('inf'), float('-inf')], 0).fillna(0).values
_, attn_weights = trainer.predict(last_seq)
axes[1].bar(range(len(attn_weights)), attn_weights, color='#EF9F27', alpha=0.7)
axes[1].set_title('Temporal Attention Weights (last 60 days)')
axes[1].set_xlabel('Time step (0=oldest, 59=most recent)')
axes[1].set_ylabel('Attention weight')

plt.tight_layout()
plt.savefig('../docs/images/lstm_training.png', bbox_inches='tight')
plt.show()
print("Higher attention weight = model focuses more on that time step")

## 7. Run Full Training Pipeline

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '../src/models/train.py', '--model', 'all', '--instrument', 'brent'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[:500])